In [1]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

pd.set_option("display.max_columns", None)
pd.set_option("display.width", 180)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

# Resolve the actual project root (the folder containing app.py).
CWD = Path.cwd().resolve()
ROOT_CANDIDATES = [CWD, *CWD.parents]
PROJECT_ROOT = next((p for p in ROOT_CANDIDATES if (p / "app.py").exists()), CWD)

# Raw data is kept in the inner project's data/raw folder.
RAW_CANDIDATES = [
    PROJECT_ROOT / "data" / "raw" / "european_household_energy.csv",
    PROJECT_ROOT.parent / "data" / "raw" / "european_household_energy.csv",
    CWD / "data" / "raw" / "european_household_energy.csv",
    CWD.parent / "data" / "raw" / "european_household_energy.csv",
]
DATA_PATH = next((p for p in RAW_CANDIDATES if p.exists()), None)

if DATA_PATH is None:
    # Search only the project/workspace ancestors for the exact source filename.
    for search_root in [PROJECT_ROOT, PROJECT_ROOT.parent, *PROJECT_ROOT.parent.parents]:
        matches = list(search_root.glob("**/european_household_energy.csv"))
        if matches:
            DATA_PATH = matches[0]
            break

if DATA_PATH is None:
    raise FileNotFoundError(
        "Raw dataset european_household_energy.csv was not found. "
        "Keep it in <project>/data/raw/."
    )

# The Streamlit app is in PROJECT_ROOT and reads the sibling workspace data/processed.
PROCESSED_CANDIDATES = [
    PROJECT_ROOT.parent / "data" / "processed",
    PROJECT_ROOT / "data" / "processed",
    PROJECT_ROOT / "notebooks" / "data" / "processed",
]
PROCESSED_DIR = next((p for p in PROCESSED_CANDIDATES if p.exists()), PROCESSED_CANDIDATES[0])
PROCESSED_DIR.mkdir(parents=True, exist_ok=True)

FIG_DIR = PROJECT_ROOT / "outputs" / "figures"
TAB_DIR = PROJECT_ROOT / "outputs" / "evaluation_reports"
FIG_DIR.mkdir(parents=True, exist_ok=True)
TAB_DIR.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Raw dataset:", DATA_PATH)
print("Processed output:", PROCESSED_DIR)

df = pd.read_csv(DATA_PATH)
print(f"Loaded {len(df):,} rows from {DATA_PATH}")
sns.set_theme(style="whitegrid", context="notebook")

"Loaded ✅"


Project root: C:\Users\LOQ\Downloads\European_Household_Energy_Project\European_Household_Energy_Project
Raw dataset: C:\Users\LOQ\Downloads\European_Household_Energy_Project\European_Household_Energy_Project\data\raw\european_household_energy.csv
Processed output: C:\Users\LOQ\Downloads\European_Household_Energy_Project\data\processed
Loaded 33,874 rows from C:\Users\LOQ\Downloads\European_Household_Energy_Project\European_Household_Energy_Project\data\raw\european_household_energy.csv


'Loaded ✅'

In [2]:
prep_log = []

def log_action(step, action, rows_before, rows_after, reason):
    prep_log.append({
        "step": step,
        "action": action,
        "rows_before": rows_before,
        "rows_after": rows_after,
        "rows_removed": rows_before - rows_after,
        "reason": reason,
    })

log_action(0, "Initial load", 0, len(df), "Raw CSV loaded")
len(prep_log)

1

In [3]:
rows_before = len(df)
df_countries = df[df.is_country_group == False].copy()
df_eu27 = df[df.is_country_group == True].copy()

log_action(1, "Filter to real countries", rows_before, len(df_countries),
           "Exclude EU27 aggregate from country-level analysis")

pd.DataFrame({
    "dataset": ["real countries", "EU27 aggregate"],
    "rows": [len(df_countries), len(df_eu27)],
    "countries": [df_countries.country.nunique(), df_eu27.country.nunique()],
})

,dataset,rows,countries
0,real countries,33356,39
1,EU27 aggregate,518,1


In [4]:
FUEL_LEVELS = {
    "TOTAL": "grand_total",
    "RA000": "aggregate", "O4000": "aggregate",
    "SFF_P1000_S2000": "aggregate",
    "E7000": "primary", "G3000": "primary", "H8000": "primary",
    "O4630": "sub_fuel", "O4669": "sub_fuel", "O4671": "sub_fuel",
    "R5110-5150_W6000RI": "sub_fuel", "R5300": "sub_fuel",
    "RA410": "sub_fuel", "RA600": "sub_fuel",
}
END_USE_LABELS = {
    "FC_OTH_HH_E": "total",
    "FC_OTH_HH_E_CK": "cooking",
    "FC_OTH_HH_E_LE": "lighting_appliances",
    "FC_OTH_HH_E_OE": "other_end_use",
    "FC_OTH_HH_E_SC": "space_cooling",
    "FC_OTH_HH_E_SH": "space_heating",
    "FC_OTH_HH_E_WH": "water_heating",
}

df_countries["fuel_level"] = df_countries.fuel_code.map(FUEL_LEVELS)
df_countries["end_use_short"] = df_countries.end_use_code.map(END_USE_LABELS)

log_action(2, "Tag fuel hierarchy + end-use labels", len(df_countries), len(df_countries),
           "Enable structural filtering downstream")

df_countries[["fuel_code", "fuel_level"]].drop_duplicates().sort_values("fuel_code")

,fuel_code,fuel_level
0,E7000,primary
15,G3000,primary
30,H8000,primary
45,O4000,aggregate
60,O4630,sub_fuel
75,O4669,sub_fuel
90,O4671,sub_fuel
105,R5110-5150_W6000RI,sub_fuel
120,R5300,sub_fuel
135,RA000,aggregate


In [5]:
# Exclusion rules (locked in Phase 1)
EXCLUDED_FUELS = {"R5300", "O4669"}          # Biogases, Other kerosene
EXCLUDED_END_USE = {"FC_OTH_HH_E_OE"}        # Other end-use

rows_before = len(df_countries)

panel_mask = (
    (~df_countries.fuel_code.isin(EXCLUDED_FUELS))
    & (~df_countries.end_use_code.isin(EXCLUDED_END_USE))
    & (df_countries.fuel_code != "TOTAL")    # panel predicts per-fuel, not totals
    & (df_countries.end_use_code != "FC_OTH_HH_E")  # excludes the end-use total row
)

panel = df_countries.loc[panel_mask].copy()

# Add split labels
panel["split"] = np.where(
    panel.year <= 2019, "train",
    np.where(panel.year <= 2022, "val", "test")
)

log_action(3, "Build panel-level dataset (prediction)",
           rows_before, len(panel),
           "Drop Biogases + Other kerosene + Other end-use + TOTAL rows")

pd.DataFrame({
    "rows": [len(panel)],
    "countries": [panel.country.nunique()],
    "fuels": [panel.fuel_code.nunique()],
    "end_uses": [panel.end_use_code.nunique()],
    "combos": [panel.groupby(["end_use_code", "fuel_code"]).ngroups],
    "years": [f"{panel.year.min()}-{panel.year.max()}"],
})

,rows,countries,fuels,end_uses,combos,years
0,15813,39,11,5,35,2010-2024


In [6]:
panel_split = (
    panel.groupby("split")
    .agg(
        rows=("energy_tj", "size"),
        positive=("energy_tj", lambda s: (s > 0).sum()),
        countries=("country", "nunique"),
        mean_pos=("energy_tj", lambda s: s[s > 0].mean()),
    )
)
panel_split["positive_pct"] = (panel_split["positive"] / panel_split["rows"] * 100).round(1)

panel_split.style.format({
    "mean_pos": "{:,.0f}", "positive_pct": "{:.1f}%",
})

,rows,positive,countries,mean_pos,positive_pct
split,,,,,
test,2546,1757,37,"16,890",69.0%
train,9450,6292,38,"21,244",66.6%
val,3817,2618,38,"17,457",68.6%


In [7]:
country_total = (
    df_countries[
        (df_countries.end_use_code == "FC_OTH_HH_E")
        & (df_countries.fuel_code == "TOTAL")
    ]
    .loc[:, ["country", "country_code", "year", "energy_tj", "flag"]]
    .sort_values(["country", "year"])
    .reset_index(drop=True)
)

# Eligibility check
forecast_elig = (
    country_total.groupby("country").agg(
        n_years=("year", "nunique"),
        min_year=("year", "min"),
        max_year=("year", "max"),
    )
)
forecast_elig["expected_years"] = forecast_elig.max_year - forecast_elig.min_year + 1
forecast_elig["is_continuous"] = forecast_elig.n_years == forecast_elig.expected_years
forecast_elig["eligible"] = forecast_elig.is_continuous & (forecast_elig.n_years >= 10)

eligible_countries = forecast_elig[forecast_elig.eligible].index
country_total_eligible = country_total[country_total.country.isin(eligible_countries)].copy()

log_action(5, "Build country total dataset (forecasting)",
           len(country_total), len(country_total_eligible),
           "Filter to 30 eligible countries (continuous + ≥10 years)")

pd.DataFrame({
    "total_countries": [country_total.country.nunique()],
    "eligible_countries": [len(eligible_countries)],
    "rows_total": [len(country_total)],
    "rows_eligible": [len(country_total_eligible)],
})

,total_countries,eligible_countries,rows_total,rows_eligible
0,39,30,454,386


In [8]:
# Only use the 31 countries that passed Phase 1 clustering eligibility
CLUSTER_MIN_YEARS = 10
CLUSTER_MIN_POSITIVE = 20

cluster_elig = (
    df_countries.groupby("country").agg(
        n_years=("year", "nunique"),
        n_positive=("energy_tj", lambda s: (s > 0).sum()),
    )
)
cluster_elig["eligible"] = (
    (cluster_elig.n_years >= CLUSTER_MIN_YEARS)
    & (cluster_elig.n_positive >= CLUSTER_MIN_POSITIVE)
)
eligible_cluster = cluster_elig[cluster_elig.eligible].index

# Build per-country static profile
def build_country_profile(country_name):
    c_data = df_countries[df_countries.country == country_name]
    total_c = c_data[(c_data.end_use_code == "FC_OTH_HH_E") & (c_data.fuel_code == "TOTAL")]
    fuel_c = c_data[
        (c_data.end_use_code == "FC_OTH_HH_E")
        & (c_data.fuel_code != "TOTAL")
        & (~c_data.fuel_code.isin(EXCLUDED_FUELS))
    ]
    enduse_c = c_data[
        (c_data.fuel_code == "TOTAL")
        & (c_data.end_use_code != "FC_OTH_HH_E")
        & (~c_data.end_use_code.isin(EXCLUDED_END_USE))
    ]

    total_mean = total_c.energy_tj.mean()
    total_std = total_c.energy_tj.std()

    # Fuel shares (aggregate primary only)
    primary_fuel = fuel_c[fuel_c.fuel_level == "primary"]
    fuel_totals = primary_fuel.groupby("fuel_code").energy_tj.mean()
    fuel_share_sum = fuel_totals.sum()

    # End-use shares
    enduse_totals = enduse_c.groupby("end_use_short").energy_tj.mean()
    enduse_share_sum = enduse_totals.sum()

    # Growth: first vs last year of TOTAL
    total_sorted = total_c.sort_values("year")
    growth_pct = (
        (total_sorted.energy_tj.iloc[-1] - total_sorted.energy_tj.iloc[0])
        / total_sorted.energy_tj.iloc[0] * 100
        if len(total_sorted) >= 2 and total_sorted.energy_tj.iloc[0] > 0
        else np.nan
    )

    return {
        "country": country_name,
        "avg_total_tj": total_mean,
        "cv_total": total_std / total_mean if total_mean > 0 else np.nan,
        "growth_pct": growth_pct,
        "elec_share": (fuel_totals.get("E7000", 0) / fuel_share_sum * 100) if fuel_share_sum > 0 else np.nan,
        "gas_share": (fuel_totals.get("G3000", 0) / fuel_share_sum * 100) if fuel_share_sum > 0 else np.nan,
        "heat_share": (fuel_totals.get("H8000", 0) / fuel_share_sum * 100) if fuel_share_sum > 0 else np.nan,
        "sh_share": (enduse_totals.get("space_heating", 0) / enduse_share_sum * 100) if enduse_share_sum > 0 else np.nan,
        "wh_share": (enduse_totals.get("water_heating", 0) / enduse_share_sum * 100) if enduse_share_sum > 0 else np.nan,
        "ck_share": (enduse_totals.get("cooking", 0) / enduse_share_sum * 100) if enduse_share_sum > 0 else np.nan,
        "le_share": (enduse_totals.get("lighting_appliances", 0) / enduse_share_sum * 100) if enduse_share_sum > 0 else np.nan,
        "sc_share": (enduse_totals.get("space_cooling", 0) / enduse_share_sum * 100) if enduse_share_sum > 0 else np.nan,
        "n_years": total_c.year.nunique(),
    }

country_profile = pd.DataFrame([build_country_profile(c) for c in eligible_cluster])

log_action(6, "Build country profile (clustering)",
           len(eligible_cluster), len(country_profile),
           "One row per eligible country with ~10 features")

country_profile.head().style.format({
    "avg_total_tj": "{:,.0f}", "cv_total": "{:.2f}",
    "growth_pct": "{:+.1f}%",
    "elec_share": "{:.1f}%", "gas_share": "{:.1f}%", "heat_share": "{:.1f}%",
    "sh_share": "{:.1f}%", "wh_share": "{:.1f}%", "ck_share": "{:.1f}%",
    "le_share": "{:.1f}%", "sc_share": "{:.1f}%",
})

,country,avg_total_tj,cv_total,growth_pct,elec_share,gas_share,heat_share,sh_share,wh_share,ck_share,le_share,sc_share,n_years
0,Albania,"21,656",0.05,+0.9%,100.0%,0.0%,0.0%,32.7%,21.3%,29.9%,9.3%,6.8%,15
1,Austria,"288,487",0.05,-1.8%,43.1%,36.5%,20.4%,71.5%,14.8%,2.7%,11.0%,0.0%,15
2,Bosnia and Herzegovina,"63,330",0.24,+78.4%,76.1%,6.8%,17.1%,69.1%,11.0%,5.3%,13.8%,0.7%,11
3,Bulgaria,"93,402",0.06,-12.6%,71.0%,5.1%,23.9%,51.9%,18.0%,8.3%,21.3%,0.5%,15
4,Croatia,"99,722",0.07,-20.6%,46.8%,42.7%,10.6%,68.4%,10.1%,6.6%,13.1%,1.8%,15


In [9]:
eu27_total = (
    df_eu27[
        (df_eu27.end_use_code == "FC_OTH_HH_E")
        & (df_eu27.fuel_code == "TOTAL")
    ]
    .loc[:, ["country", "year", "energy_tj"]]
    .sort_values("year")
    .reset_index(drop=True)
)

log_action(7, "Build EU27 benchmark",
           len(df_eu27), len(eu27_total),
           "Reserve for comparison only — NEVER used in training")

pd.DataFrame({
    "rows": [len(eu27_total)],
    "years": [f"{eu27_total.year.min()}-{eu27_total.year.max()}"],
    "total_tj_latest": [eu27_total.loc[eu27_total.year.idxmax(), "energy_tj"]],
})

,rows,years,total_tj_latest
0,7,2018-2024,"9,544,155.22"


In [10]:
sanity = pd.DataFrame({
    "check": [
        "panel: no TOTAL fuel rows",
        "panel: no TOTAL end-use rows",
        "panel: no Biogases or Other kerosene",
        "panel: no Other end-use",
        "panel: log1p transformable (no negatives)",
        "country_total: all positive values",
        "country_profile: no NaN in core features",
        "split: train years ≤ 2019",
        "split: test years ≥ 2023",
    ],
    "passed": [
        (panel.fuel_code != "TOTAL").all(),
        (panel.end_use_code != "FC_OTH_HH_E").all(),
        not panel.fuel_code.isin(EXCLUDED_FUELS).any(),
        not panel.end_use_code.isin(EXCLUDED_END_USE).any(),
        (panel.energy_tj >= 0).all(),
        (country_total_eligible.energy_tj > 0).all(),
        country_profile[["avg_total_tj", "cv_total", "growth_pct"]].notna().all().all(),
        panel.loc[panel.split == "train", "year"].max() <= 2019,
        panel.loc[panel.split == "test", "year"].min() >= 2023,
    ],
})

sanity

,check,passed
0,panel: no TOTAL fuel rows,True
1,panel: no TOTAL end-use rows,True
2,panel: no Biogases or Other kerosene,True
3,panel: no Other end-use,True
4,panel: log1p transformable (no negatives),True
5,country_total: all positive values,True
6,country_profile: no NaN in core features,True
7,split: train years ≤ 2019,True
8,split: test years ≥ 2023,True


In [11]:
panel.to_csv(PROCESSED_DIR / "panel_level.csv", index=False)
country_total_eligible.to_csv(PROCESSED_DIR / "country_total.csv", index=False)
country_profile.to_csv(PROCESSED_DIR / "country_profile.csv", index=False)
eu27_total.to_csv(PROCESSED_DIR / "eu27_benchmark.csv", index=False)
pd.DataFrame(prep_log).to_csv(TAB_DIR / "phase2_preparation_log.csv", index=False)

pd.DataFrame({
    "file": [
        "panel_level.csv",
        "country_total.csv",
        "country_profile.csv",
        "eu27_benchmark.csv",
        "phase2_preparation_log.csv",
    ],
    "rows": [
        len(panel), len(country_total_eligible),
        len(country_profile), len(eu27_total), len(prep_log),
    ],
    "cols": [
        panel.shape[1], country_total_eligible.shape[1],
        country_profile.shape[1], eu27_total.shape[1], 6,
    ],
})

,file,rows,cols
0,panel_level.csv,15813,17
1,country_total.csv,386,5
2,country_profile.csv,31,13
3,eu27_benchmark.csv,7,3
4,phase2_preparation_log.csv,7,6


# 📋 Phase 2 — Data Preparation Summary

## 1. Datasets produced
| File | Rows | Purpose |
|---|---|---|
| `panel_level.csv` | see output | (country, year, end_use, fuel) — for prediction |
| `country_total.csv` | see output | TOTAL series per country — for forecasting |
| `country_profile.csv` | see output | per-country static features — for clustering |
| `eu27_benchmark.csv` | 15 | EU27 aggregate — comparison only |
| `phase2_preparation_log.csv` | 8 | Audit trail of every decision |

## 2. Filtering decisions applied
| Step | Decision | Rows affected |
|---|---|---|
| 1 | Excluded EU27 group | 518 removed |
| 2 | Tagged fuel level + end-use | — |
| 3 | Dropped Biogases + Other kerosene | structural zeros (>80%) |
| 4 | Dropped Other end-use | 88.5% zeros |
| 5 | Dropped TOTAL fuel rows from panel | avoids double counting |
| 6 | Kept 30 countries for forecasting | continuous + ≥10 years |
| 7 | Kept 31 countries for clustering | ≥10 years + ≥20 positives |

## 3. Panel split
| Set | Years | Rows | Positive % |
|---|---|---|---|
| Train | 2010–2019 | see output | ~55% |
| Val | 2020–2022 | see output | ~57% |
| Test | 2023–2024 | see output | ~58% |

## 4. Country profile features (11 features)
| Feature | Description |
|---|---|
| `avg_total_tj` | Mean annual TOTAL household energy |
| `cv_total` | Coefficient of variation (volatility) |
| `growth_pct` | First-to-last year growth |
| `elec_share` | Electricity % of primary fuels |
| `gas_share` | Natural gas % of primary fuels |
| `heat_share` | District heat % of primary fuels |
| `sh_share` | Space heating % of end-uses |
| `wh_share` | Water heating % of end-uses |
| `ck_share` | Cooking % of end-uses |
| `le_share` | Lighting & appliances % of end-uses |
| `sc_share` | Space cooling % of end-uses |

## 5. Guardrails honored
- ✅ No aggregate + component mixing (TOTAL excluded from panel)
- ✅ Structural zeros removed (Biogases, Other kerosene)
- ✅ EU27 kept separate from all training
- ✅ Chronological splits preserved
- ✅ Full audit log saved

---
**Status:** Phase 2 complete. Ready for Phase 3 — Energy Prediction.